# Diffusion Transformer (DiT)

**Domain:** Architectures  ·  **recommended addition**  ·  **runnable:** yes

A refresher on the **Diffusion Transformer** — the architecture that replaced the U-Net backbone of latent
diffusion models with a plain Transformer over image-patch tokens, and showed that **diffusion scales like the
rest of deep learning** (Peebles & Xie, *Scalable Diffusion Models with Transformers*, 2022). DiT is the
backbone under Sora, Stable Diffusion 3, PixArt-α, and Flux.

## 1. What & Why

A **DiT** is the denoising network of a diffusion model built as a **Transformer instead of a U-Net**. It
borrows the Vision-Transformer recipe wholesale: take the noised image (actually its **VAE latent**, à la
latent diffusion), chop it into a grid of patches, project each patch to a token, run a stack of standard
Transformer blocks, and project back to predict the noise (or velocity). The diffusion *process* — add noise
over T steps, train a network to undo one step — is unchanged. **Only the architecture of the denoiser
changes.**

**The problem it solves.** For years the denoiser in every diffusion model was a **U-Net**: a convolutional
encoder–decoder with skip connections, plus bolted-on self-attention at a few resolutions. U-Nets work, but
they carry a pile of hand-designed inductive bias (multi-scale conv hierarchy, skip topology, where to put
attention) and they don't have a clean scaling story. Peebles & Xie asked: does the *U-Net* matter, or just
the diffusion framework? Their answer: swap it for a Transformer and you get a **single, uniform, attention-
only** backbone whose quality improves **smoothly and predictably with compute (Gflops)** — the same
scaling-law behavior that made Transformers eat NLP and vision. DiT-XL/2 beat the best U-Net diffusion models
(ADM, LDM) on ImageNet 256×256 and 512×512.

**The key architectural trick is conditioning.** A diffusion denoiser must be told *which timestep* it's at
and (for class/text-conditional generation) *what to generate*. DiT injects this conditioning through
**adaptive LayerNorm with zero-init (adaLN-Zero)**: a small MLP turns the conditioning vector into per-block
scale/shift/gate parameters, and the residual gates start at zero so **every block begins as the identity**.
This is what makes deep DiTs train stably, and it beat cross-attention and in-context conditioning in their
ablations.

**Reach for it when** you're building a generative image/video model at scale, want one architecture you can
grow with more compute, or want to share Transformer infrastructure (FlashAttention, sequence parallelism, MoE)
with your LLM stack. **Don't** reach for it when you have a tiny compute/data budget — a U-Net's conv priors
are more sample-efficient at small scale and DiT's O(N²) attention gets expensive at high latent resolution.

## 2. Mental Model

**A DiT is just a Vision Transformer that predicts noise instead of a class — with the timestep and label
whispered into every LayerNorm.**

```
   noised VAE latent              sequence of patch tokens         L × DiT blocks            predicted noise
   32×32×4  (B,4,32,32)                                          ┌────────────────────┐     32×32×4
 ┌──┬──┬──┬──┐                  [t1 t2 t3 ... t256]  + pos        │  adaLN(t,y) → γ,β,α │   ┌──┬──┬──┬──┐
 │  │  │  │  │  patchify (P=2)   each 2×2 patch → Linear → token  │  self-attention     │   │  │  │  │  │
 ├──┼──┼──┼──┤  ───────────────▶                          ──────▶ │  adaLN(t,y) → γ,β,α │──▶├──┼──┼──┼──┤
 │  │  │  │  │                                                    │  MLP                │   │  │  │  │  │
 └──┴──┴──┴──┘                                                    └────────▲───────────┘   └──┴──┴──┴──┘
   16×16 = 256 patches                                                     │  unpatchify → noise
                                          timestep t ─┐                    │
                                          class/text y┴── MLP ── c ────────┘  (conditioning, NOT a token)
```

Three things make it click:

- **It's ViT, end-to-end.** Patchify → +positional → Transformer blocks → un-patchify. If you know ViT, you
  know 90% of DiT. The two differences: the *input* is a noisy latent (not a clean image), and the *output* is
  a same-shape noise prediction (not a class logit).
- **The timestep & label are not tokens — they steer the normalization.** Instead of prepending a
  conditioning token, DiT feeds the conditioning vector `c` into a small MLP that emits **scale (γ), shift (β),
  and gate (α)** for each LayerNorm/residual in every block (`adaLN-Zero`). The network is *modulated* by the
  condition rather than attending to it.
- **Zero-init = identity at birth.** The gates α start at zero, so at initialization each block outputs its
  input unchanged and the whole DiT is the identity function. Training then gently "turns on" each block. This
  is the trick that lets you stack a lot of blocks and still converge.

## 3. Key Concepts

- **Latent diffusion** — DiT denoises in the **compressed latent space** of a pretrained VAE (e.g. a 256×256×3
  image → 32×32×4 latent, an 8× spatial / 48× volume reduction), not raw pixels. This is what makes
  Transformer attention affordable. Same idea as Stable Diffusion / LDM.
- **Patchify (patch size P)** — split the latent into P×P patches and linearly project each to a token (a
  strided `Conv2d`, exactly as in ViT). DiT names variants by patch size: **DiT-XL/2** = XL model, patch 2.
  Smaller P → more tokens → more compute and better quality. /2 was the sweet spot.
- **adaLN-Zero (the headline idea)** — *adaptive LayerNorm, zero-initialized.* An MLP maps the conditioning
  vector `c = embed(t) + embed(y)` to **6 vectors per block**: shift+scale for the attention norm, shift+scale
  for the MLP norm, and a **gate (α)** scaling each residual branch. The final linear layer is initialized to
  zero so every block starts as identity. Beat in-context conditioning and cross-attention in the paper's
  ablation, at negligible extra cost.
- **Timestep & label embeddings** — `t` becomes a sinusoidal embedding then an MLP; the class label `y` (or a
  text embedding) becomes a learned vector; they're **summed** into `c`. Classifier-free guidance works by
  dropping `y` with some probability during training.
- **Model scaling (S/B/L/XL)** — DiT scales by depth, width, and heads exactly like ViT. The central result:
  **FID drops smoothly as you add Gflops**, whether via bigger models or smaller patches — a clean scaling law
  for diffusion.
- **Output head & unpatchify** — a final adaLN + linear projects each token to `P·P·C` values, reshaped back
  to the latent grid: the predicted noise ε (or v-prediction). Loss is the usual diffusion MSE.
- **Where it went** — DiT is the lineage behind **Sora** (video, spacetime patches), **Stable Diffusion 3 /
  Flux** (**MMDiT**: parallel text+image streams with joint attention), and **PixArt-α** (cross-attention DiT
  for efficient text-to-image).

## 4. Setup

The worked examples use only **PyTorch (CPU is fine)** plus NumPy. We build patchify, an **adaLN-Zero DiT
block**, and a tiny end-to-end DiT from scratch with `torch.nn`, so the mechanics — especially the
zero-init-identity trick — are visible. No GPU, no downloads, no API keys.

The final cell shows how to pull a *real* pretrained **DiT-XL/2** via 🤗 `diffusers`; it's **gated** behind an
`os.getenv("ALLOW_DOWNLOAD")` check (it downloads weights), so the notebook runs offline either way.

In [1]:
# %pip install numpy torch   # the CPU build of torch is enough
import math, os, importlib.util
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)
print("numpy", np.__version__, "| torch", torch.__version__)

numpy 2.4.6 | torch 2.12.1


## 5. Worked Examples

### Example 1 — Patchify a latent + build the conditioning vector

The "image → tokens" front end, plus the conditioning that makes diffusion different from plain ViT. We take a
32×32×4 latent (what an SD-style VAE produces for a 256×256 image), project non-overlapping 2×2 patches to
tokens, then build the conditioning vector `c` from a **sinusoidal timestep embedding** plus a **class-label
embedding**. Watch the shapes: a 4-D latent becomes a `(batch, tokens, D)` sequence, and `c` is a single
`(batch, D)` vector — not a token.

In [2]:
# A DiT denoises the VAE *latent*, not raw pixels. SD's VAE: 256x256x3 image -> 32x32x4 latent.
B, C, H, W = 1, 4, 32, 32        # (batch, latent channels, height, width)
P = 2                            # DiT patch size on the latent  (the "/2" in DiT-XL/2)
D = 384                          # token / hidden dim

latent = torch.randn(B, C, H, W)

# Patchify exactly like ViT: a strided conv projects each PxP latent patch to one token.
patchify = nn.Conv2d(C, D, kernel_size=P, stride=P)
tokens = patchify(latent).flatten(2).transpose(1, 2)        # (B, T, D)
T = tokens.shape[1]
print(f"latent {tuple(latent.shape)} -> {T} patch tokens of dim {D}: {tuple(tokens.shape)}")

# Timestep t -> sinusoidal embedding (same construction as Transformer positions).
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

t = torch.randint(0, 1000, (B,))                            # diffusion timestep
y = torch.tensor([207])                                     # class label (e.g. ImageNet id)
cond = timestep_embedding(t, D) + nn.Embedding(1000, D)(y)  # c = embed(t) + embed(y)
print("timestep", t.tolist(), "label", y.tolist(), "-> conditioning vector c:", tuple(cond.shape))

latent (1, 4, 32, 32) -> 256 patch tokens of dim 384: (1, 256, 384)
timestep [597] label [207] -> conditioning vector c: (1, 384)


### Example 2 — The adaLN-Zero block, and why it starts as the identity

This is the heart of DiT. The conditioning vector `c` is pushed through one MLP that emits **six** modulation
vectors: shift/scale/gate for the attention sub-layer and shift/scale/gate for the MLP sub-layer. The block's
LayerNorms have *no* learnable affine params — the affine comes from `c` instead. The final linear of the
modulation MLP is **zero-initialized**, so at init the gates α are zero and **the block returns its input
unchanged**. We verify exactly that.

In [3]:
def modulate(x, shift, scale):
    # broadcast (B, D) modulation across the token axis
    return x * (1 + scale[:, None]) + shift[:, None]

class DiTBlock(nn.Module):
    """A Transformer block conditioned via adaLN-Zero."""
    def __init__(self, dim, heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.mlp = nn.Sequential(nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(),
                                 nn.Linear(int(dim * mlp_ratio), dim))
        # One MLP maps the conditioning vector c -> 6 * dim modulation params.
        self.adaLN = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        nn.init.zeros_(self.adaLN[-1].weight)   # <-- the "Zero" in adaLN-Zero
        nn.init.zeros_(self.adaLN[-1].bias)

    def forward(self, x, c):
        sh_a, sc_a, gate_a, sh_m, sc_m, gate_m = self.adaLN(c).chunk(6, dim=1)
        h = modulate(self.norm1(x), sh_a, sc_a)
        attn, _ = self.attn(h, h, h)
        x = x + gate_a[:, None] * attn                       # gated residual
        h = modulate(self.norm2(x), sh_m, sc_m)
        x = x + gate_m[:, None] * self.mlp(h)                # gated residual
        return x

block = DiTBlock(D, heads=6)
out = block(tokens, cond)
print("max |out - in| at initialization:", f"{(out - tokens).abs().max().item():.2e}")
print("-> every adaLN-Zero block is the identity at init, so deep DiTs train stably.")
print("block params:", f"{sum(p.numel() for p in block.parameters()):,}")

max |out - in| at initialization: 0.00e+00
-> every adaLN-Zero block is the identity at init, so deep DiTs train stably.
block params: 2,659,968


### Example 3 — A tiny DiT, end to end (noisy latent → predicted noise)

Assemble the full path: patchify → +positional → a stack of DiT blocks → a final adaLN layer → linear head →
**un-patchify** back to the latent grid. The output has the **same shape as the input latent** — it's the
predicted noise ε the diffusion loss compares against. This runs a real forward pass on CPU.

In [4]:
class TinyDiT(nn.Module):
    def __init__(self, in_ch=4, patch=2, dim=384, depth=4, heads=6, n_classes=1000, grid=16):
        super().__init__()
        self.patch, self.in_ch, self.grid, self.dim = patch, in_ch, grid, dim
        self.x_embed = nn.Conv2d(in_ch, dim, patch, patch)
        self.pos = nn.Parameter(torch.randn(1, grid * grid, dim) * 0.02)
        self.t_mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.y_embed = nn.Embedding(n_classes, dim)
        self.blocks = nn.ModuleList([DiTBlock(dim, heads) for _ in range(depth)])
        self.norm_final = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.adaLN_final = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))
        nn.init.zeros_(self.adaLN_final[-1].weight); nn.init.zeros_(self.adaLN_final[-1].bias)
        self.head = nn.Linear(dim, patch * patch * in_ch)    # predict noise per patch

    def forward(self, x, t, y):
        x = self.x_embed(x).flatten(2).transpose(1, 2) + self.pos          # (B, T, dim)
        c = self.t_mlp(timestep_embedding(t, self.dim)) + self.y_embed(y)  # (B, dim)
        for blk in self.blocks:
            x = blk(x, c)
        shift, scale = self.adaLN_final(c).chunk(2, dim=1)
        x = modulate(self.norm_final(x), shift, scale)
        x = self.head(x)                                                   # (B, T, P*P*C)
        # un-patchify: (B, T, P*P*C) -> (B, C, H, W)
        g, p, ch = self.grid, self.patch, self.in_ch
        x = x.reshape(x.shape[0], g, g, p, p, ch).permute(0, 5, 1, 3, 2, 4)
        return x.reshape(x.shape[0], ch, g * p, g * p)

model = TinyDiT()
noise_pred = model(latent, t, y)
print("noisy latent:", tuple(latent.shape), "-> predicted noise:", tuple(noise_pred.shape))
print("params:", f"{sum(p.numel() for p in model.parameters()):,}  (this is a toy; DiT-XL/2 is ~675M)")

noisy latent: (1, 4, 32, 32) -> predicted noise: (1, 4, 32, 32)
params: 11,726,224  (this is a toy; DiT-XL/2 is ~675M)


### Example 4 — Patch size is the compute lever (and why DiT works in latent space)

Attention is **O(N²)** in the token count N, and N = (latent_size / patch)². Halving the patch quadruples N
and ~16×'s attention cost — the same wall ViT hits. The table also shows *why DiT denoises latents, not
pixels*: at patch 2, a 256×256 **pixel** grid would be 16,384 tokens, but its 32×32 **latent** is only 256.

In [5]:
print(f"{'space':>16} | {'grid':>9} | {'patch':>5} | {'tokens N':>8} | {'attn ~ N^2':>12}")
print("-" * 64)
rows = [
    ("pixels 256x256", 256, 2),
    ("latent 32x32",    32, 8),
    ("latent 32x32",    32, 4),
    ("latent 32x32",    32, 2),   # DiT-*/2
    ("latent 64x64",    64, 2),   # 512px images
]
for name, size, patch in rows:
    n = (size // patch) ** 2
    print(f"{name:>16} | {size:>4}x{size:<4} | {patch:>4}px | {n:>8,} | {n * n:>12,}")
print("\nWorking in the VAE latent (32x32) instead of pixels (256x256) cuts tokens ~64x at the same")
print("patch size -- that is what makes pure-attention diffusion affordable. Smaller patch = better/costlier.")

           space |      grid | patch | tokens N |   attn ~ N^2
----------------------------------------------------------------
  pixels 256x256 |  256x256  |    2px |   16,384 |  268,435,456
    latent 32x32 |   32x32   |    8px |       16 |          256
    latent 32x32 |   32x32   |    4px |       64 |        4,096
    latent 32x32 |   32x32   |    2px |      256 |       65,536
    latent 64x64 |   64x64   |    2px |    1,024 |    1,048,576

Working in the VAE latent (32x32) instead of pixels (256x256) cuts tokens ~64x at the same
patch size -- that is what makes pure-attention diffusion affordable. Smaller patch = better/costlier.


In [6]:
# OPTIONAL: a real pretrained DiT-XL/2 via diffusers (downloads ~2.7GB of weights).
# Gated behind ALLOW_DOWNLOAD so the notebook runs offline; shows the call shape.
if importlib.util.find_spec("diffusers") is not None and os.getenv("ALLOW_DOWNLOAD"):
    from diffusers import DiTPipeline

    pipe = DiTPipeline.from_pretrained("facebook/DiT-XL-2-256")     # class-conditional ImageNet
    image = pipe(class_labels=[207], num_inference_steps=25).images[0]   # 207 = golden retriever
    print("DiT-XL/2 generated an image of size:", image.size)
else:
    print("Skipping pretrained DiT (set ALLOW_DOWNLOAD=1 and `pip install diffusers` to run it).")
    print('Shape: DiTPipeline.from_pretrained("facebook/DiT-XL-2-256")(class_labels=[207]).images[0]')

Skipping pretrained DiT (set ALLOW_DOWNLOAD=1 and `pip install diffusers` to run it).
Shape: DiTPipeline.from_pretrained("facebook/DiT-XL-2-256")(class_labels=[207]).images[0]


## 6. Gotchas & Pitfalls

- **Denoise the latent, not pixels.** DiT-style models assume a pretrained VAE and operate on its latent.
  Feeding raw pixels makes the token count explode (16k+ tokens at patch 2 for 256px) and the math/normalization
  won't match. The VAE's scaling factor (e.g. SD's `0.18215`) must be applied — forget it and samples are
  washed out or saturated.
- **adaLN-Zero must actually be zero-initialized.** The whole stability story rests on the *final* linear of
  each modulation MLP starting at zero (so blocks begin as identity). Initialize it normally and a deep DiT
  trains far worse — this is the single most common reimplementation bug.
- **LayerNorm should have no affine params.** Use `elementwise_affine=False`; the scale/shift come from `c`
  via adaLN. Leaving the built-in affine on double-counts the modulation and hurts.
- **Patch size sets cost *and* quality, quadratically.** Going from /4 to /2 roughly 16×'s attention FLOPs.
  Don't shrink the patch (or raise latent resolution) without checking the compute budget; conversely, /2
  beats /4 and /8 on quality in the paper.
- **Positional handling is fixed to one grid.** Learned (or sin-cos) positions are tied to the token grid;
  changing latent resolution needs interpolation/extrapolation (2-D RoPE or sin-cos extrapolation in newer
  variants), or the model breaks.
- **Conditioning choice matters.** adaLN-Zero won for **class** labels; for rich **text** conditioning,
  practical systems add **cross-attention** (PixArt-α) or joint text+image attention (**MMDiT** in SD3). Don't
  assume adaLN alone is enough for text-to-image.
- **DiT is data/compute hungry at the low end.** Like ViT vs CNN, a small-budget DiT can lose to a tuned
  U-Net. The scaling-law payoff shows up with scale; don't benchmark a tiny DiT and conclude "Transformers
  don't help diffusion."
- **It's a denoiser, not a sampler.** DiT only predicts noise/velocity for one step. You still need a sampler
  (DDPM/DDIM/flow-matching ODE) and, usually, classifier-free guidance to generate images.

## 7. When to Use vs Alternatives

| Backbone | Inductive bias | Cost vs resolution | Scaling story | Best when |
|---|---|---|---|---|
| **DiT (adaLN-Zero)** | Minimal; global attention from layer 1 | O(N²) in latent tokens | **Clean** — FID ↓ smoothly with Gflops | Scaling up; class/simple conditioning; sharing Transformer infra |
| **U-Net (ADM/LDM)** | Strong: conv multi-scale + skips | O(pixels/latent) — gentle | Murkier; lots of hand-tuning | Small/medium compute; sample-efficiency matters; mature tooling |
| **MMDiT (SD3 / Flux)** | DiT + joint text–image attention | O(N²), two streams | Same as DiT, strong for text | **Text-to-image** at scale; best current open T2I quality |
| **PixArt-α (cross-attn DiT)** | DiT + cross-attention to text | O(N²) + cross-attn | Efficient — trains cheaply | T2I on a budget; reuse a frozen text encoder |
| **U-ViT / hybrid** | ViT with U-Net-style long skips | O(N²) | Between the two | Want Transformer + skip-connection sample efficiency |

**Rules of thumb.** Building a large, scalable generative model and want one architecture that grows
predictably with compute and rides Transformer tooling? **DiT.** Doing **text-to-image** specifically? Use a
DiT *variant* with proper text conditioning — **MMDiT** (SD3/Flux) for top quality, **PixArt-α** for
efficiency — not vanilla class-conditional DiT. Tight on compute/data, or you already have a battle-tested
U-Net pipeline? A **U-Net** is more sample-efficient at small scale and has the most mature ecosystem. Want
Transformer scalability *plus* the U-Net's skip-connection sample efficiency? Look at **U-ViT**.

## 8. Resources

- **DiT paper** — Peebles & Xie, *Scalable Diffusion Models with Transformers* (2022): https://arxiv.org/abs/2212.09748
- **Official code & checkpoints** — `facebookresearch/DiT`: https://github.com/facebookresearch/DiT
- **Project page (samples, intuition)** — https://www.wpeebles.com/DiT
- **🤗 diffusers DiT pipeline** — pretrained DiT-XL/2 in two lines: https://huggingface.co/docs/diffusers/api/pipelines/dit
- **MMDiT / Stable Diffusion 3** — Esser et al., *Scaling Rectified Flow Transformers* (2024): https://arxiv.org/abs/2403.03206
- **PixArt-α** — efficient cross-attention DiT for text-to-image (2023): https://arxiv.org/abs/2310.00426
- **Sora technical report** — DiT on spacetime patches for video: https://openai.com/research/video-generation-models-as-world-simulators